In [ ]:
from pathlib import Path
import pandas as pd
import polars as pl
 
from credit_risk.data.ingestion import load_raw_accepted_loans
from credit_risk.data.target import build_target
from credit_risk.features.build_dataset import assemble_feature_matrix, gbm_features
from credit_risk.features.woe import WOEEncoder, rank_features_by_iv, prune_correlated_features
from credit_risk.evaluation.diagnostics import coefficient_sign_report, multicollinearity_report
from sklearn.linear_model import LogisticRegression

In [ ]:
DATA_PATH = Path("../data/raw/accepted_2007_to_2018Q4.csv")
df = load_raw_accepted_loans(DATA_PATH)
labeled = build_target(df)
final = assemble_feature_matrix(labeled, Path("../configs/base.yaml"))
train = final.filter(pl.col("split") == "train")

print(train.shape)

### Candidate Pool

In [ ]:
candidates = gbm_features(final)
print(f"{len(candidates)} candidate features")

### IV Ranking

In [ ]:
iv_ranked = rank_features_by_iv(train, candidates, n_bins=10).sort("iv", descending=True)
print(iv_ranked.to_pandas().to_string(index=False))
 
above_threshold = iv_ranked.filter(pl.col("iv") >= 0.02)["feature"].to_list()
print(f"{len(above_threshold)} features with IV >= 0.02")

### Preliminary fit + diagnostics (exploratory - this model is thrown away)

In [ ]:
encoder = WOEEncoder(features=above_threshold, n_bins=10).fit(train)
train_woe = encoder.transform(train)
woe_cols = [f"{f}_woe" for f in above_threshold]
prelim_model = LogisticRegression(max_iter=1000).fit(
    train_woe.select(woe_cols).to_pandas(), train_woe["default_flag"].to_pandas()
)

collinearity = multicollinearity_report(train_woe, above_threshold, threshold=0.6)

print("Coefficient signs (all should be negative - see WOEEncoder docstring for why):")
print(coefficient_sign_report(prelim_model, above_threshold).to_string(index=False))
print("Multicollinearity (|correlation| > 0.6):")
print(collinearity.to_string(index=False) if len(collinearity) else "none found")

### Prune correlated features

In [ ]:
corr_for_pruning = train_woe.select(woe_cols).to_pandas()
corr_for_pruning.columns = [c.replace("_woe", "") for c in corr_for_pruning.columns]
final_features = prune_correlated_features(above_threshold, corr_for_pruning.corr(), threshold=0.6)
print(f"{len(final_features)} final features (was {len(above_threshold)})")
print(final_features)

### Re-check after pruning

In [ ]:
encoder2 = WOEEncoder(features=final_features, n_bins=10).fit(train)
train_woe2 = encoder2.transform(train)
print(multicollinearity_report(train_woe2, final_features, threshold=0.6))